[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PedroLormendez/jcclass/blob/main/notebooks/tutorial_sample_data.ipynb)

# jcclass — Tutorial using sample data

This notebook demonstrates how to use `jcclass` to compute Jenkinson-Collison Circulation Types using the pre-loaded ERA5 mean sea-level pressure (MSLP) data provided in the `sample_data/` folder.

In [1]:
import matplotlib.pyplot as plt
!pip install jcclass
# Install ipywidgets dependencies (external, not included in jcclass)
!pip install ipywidgets



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 1. Load sample MSLP data

In [2]:
import xarray as xr

ds = xr.open_dataset('../sample_data/era5_hourly_highres.nc')

# Extract the mean sea-level pressure variable and rename to match jcclass convention
mslp = ds['msl'].rename('mean_sea_level_pressure')
mslp

<xarray.DataArray 'mean_sea_level_pressure' (time: 7, latitude: 721,
                                             longitude: 1440)> Size: 58MB
[7267680 values with dtype=float64]
Coordinates:
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * time       (time) datetime64[ns] 56B 2012-10-26T12:00:00 ... 2012-10-26T1...
Attributes:
    units:          Pa
    long_name:      Mean sea level pressure
    standard_name:  air_pressure_at_mean_sea_level

## 2. Compute Jenkinson-Collison Circulation Types

`compute_cts()` returns the **full 27-type classification** (values 0–26), covering all combinations of cyclonic/anticyclonic centres and their wind flow directions. The 27 types are the raw output of the Jenkinson-Collison algorithm.

In [3]:
from jcclass import compute_cts

cts = compute_cts(mslp)

100%|█████████████████████████████████████████████████████| 6/6 , Done ✓


## 3. Plot the results

> **Note:** `plot_cts()` displays the **11 reduced circulation types** (aggregated from the 27 original types into: LF, A, NE, E, SE, S, SW, W, NW, N, C). This is a simplified view designed for visualisation. The full 27-type data is preserved in the `cts` DataArray returned by `compute_cts()`.

By default `plot_cts()` shows the full global domain. You can zoom into any region using the `lat_south`, `lat_north`, `lon_west` and `lon_east` arguments.

Pick a timestamp from the dropdown and click **Plot** to render the circulation type map for that time step.

In [9]:
from jcclass import plot_cts
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# One dropdown entry per available timestamp in `cts`
time_options = [(str(t)[:19], t) for t in cts.time.values]

time_dropdown = widgets.Dropdown(
    options=time_options,
    description='Timestamp:',
    style={'description_width': 'initial'},
)
plot_button = widgets.Button(description='Plot', button_style='primary')
output = widgets.Output()

def on_plot_clicked(_):
    with output:
        output.clear_output(wait=True)
        fig = plot_cts(cts.sel(time=time_dropdown.value), show=True)
        #fig.savefig("figure.png", dpi=150, bbox_inches='tight')

plot_button.on_click(on_plot_clicked)
display(widgets.HBox([time_dropdown, plot_button]), output)

Output()